In [ ]:
import os
import sys

if os.getcwd().split('/')[-1] == 'genjoin':
    os.chdir('../../')
os.getcwd()

import random
import math
import numpy as np
import pandas as pd
import networkx as nx
import argparse
import itertools
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.masked import masked_tensor, as_masked_tensor
from torch.optim.lr_scheduler import ExponentialLR, StepLR
from torch.utils.data import random_split
from scipy import stats
from sklearn.model_selection import train_test_split

from src.database.postgres_executor import PostgresExecutor
from src.database.query_loader import QueryLoader
from src.database.utils import PlanParser
from src.database.selectivity_extractor import SelectivityExtractorCannotExtractException
from src.encoding.encoder import PSTHistogramEncoder, PSTEncoder, positional_encoding
from src.encoding.join_graph import JoinGraph, JoinType, RedundantJoinException, JoinNotPossibleException
from src.database.utils import ParsingNotPossible
from src.models.query_environment import QueryEnv, TIMEOUT_IN_SEC
from src.models.cvae import CVAE_G
from src.utils.query_helper import parse_sql_query
# from src.database.utils import SchemaExtractor
from src.utils.path_helper import rl_model_folder_path, WORKLOADS
from src.utils.train_test_splits import get_train_test_split
from src.encoding.database_graph import DatabaseGraph

from tqdm.auto import tqdm
import wandb

import warnings
warnings.filterwarnings(action='ignore', category=UserWarning)
warnings.filterwarnings(action='error', category=RuntimeWarning)

import optuna
from optuna.trial import TrialState

%load_ext autoreload
%autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
LOGS_FILENAME = 'logs'
TIMEOUT = 1.5 * 60 * 1000
N_RUNS = 3
EXPERIMENTS = {
    'LeaveOneOutSplit1': 'leave_one_out_split_1', 
    'LeaveOneOutSplit2': 'leave_one_out_split_2',
    'LeaveOneOutSplit3': 'leave_one_out_split_3',
    'RandomSplit1': 'random_split_1',
    'RandomSplit2': 'random_split_2',
    'RandomSplit3': 'random_split_3',
    'BaseQuerySplit1': 'base_query_split_1', 
    'BaseQuerySplit2': 'base_query_split_2',
    'BaseQuerySplit3': 'base_query_split_3',
    'LeaveOneOutSplitA': 'leave_one_out_split_a', 
    'LeaveOneOutSplitB': 'leave_one_out_split_b',
    'LeaveOneOutSplitC': 'leave_one_out_split_c',
    'RandomSplitA': 'random_split_a',
    'RandomSplitB': 'random_split_b',
    'RandomSplitC': 'random_split_c',
    'BaseQuerySplitA': 'base_query_split_a', 
    'BaseQuerySplitB': 'base_query_split_b',
    'BaseQuerySplitC': 'base_query_split_c',
    'SlowSplit': 'slow_split'}

# NUM_POSSIBLE_JOINS = 53  JOB: 53, STACK: 32
NUM_JOIN_TYPES = 3
QUERY_ENCODING_SIZE = 3

DEMO_BASE_PATH = f'data/demonstrations/'
PROCESSED_BASE_PATH = f'data/processed/'
DEMO_TYPE = 'cvae'
LOGS_FILENAME = 'logs'

device = torch.device("cuda")
softmax = nn.Softmax(dim=1)

torch.manual_seed(1234)
np.random.seed(1234)

WORKLOAD_NAME = 'stack'  #  'join_order_benchmark', 'stack'
mode = 'test'
experiment = 'SlowSplit'
pg_live = True
SCHEMA_FOLDER = WORKLOADS[WORKLOAD_NAME]
TEST_RESULTS = f"full_execution_time_{mode}.csv"

if WORKLOAD_NAME == 'join_order_benchmark':
    norm = 100
elif WORKLOAD_NAME == 'stack':
    norm = 50

print(f'Experiment: {experiment}, Mode: {amode}')
if experiment == 'join_order_benchmark':
    norm = 100
else:
    norm = 50

train_queries, _ = get_train_test_split(experiment, workload_name=WORKLOAD_NAME)
train_queries, test_queries = train_test_split(train_queries, test_size=0.1)

In [ ]:
for train_query in tqdm(train_queries, desc="Processing train queries ...", total=len(train_queries)):
    # counter = 0
    query_enc = torch.load(os.path.join(DEMO_BASE_PATH, SCHEMA_FOLDER, DEMO_TYPE, train_query, 'query_enc.pt'))
    plan_enc = torch.load(os.path.join(DEMO_BASE_PATH, SCHEMA_FOLDER, DEMO_TYPE, train_query, 'plan_enc.pt'))
    exec_times = torch.load(os.path.join(DEMO_BASE_PATH, SCHEMA_FOLDER, DEMO_TYPE, train_query, 'exec_times.pt'))
    exec_time = torch.mean(exec_times, axis=1)
    exec_time, indices = torch.sort(exec_time, descending=True)
    plan_enc = plan_enc[indices]
    
    pg_times = pg_df[pg_df.query_ident == f'{train_query}'].execution_time.values
    
    for in_id, out_id in itertools.combinations(np.arange(0, len(exec_time) - 1), 2):
        # Problem with calculating p-values when data is close to identical
        try:
            _, out_dist = stats.ttest_ind(exec_times[in_id].numpy(), exec_times[out_id].numpy(),
                                        alternative='greater')
        except RuntimeWarning:
            out_dist = 1.0
        if math.isnan(out_dist):
            out_dist = 1.0
        try:
            _, pg_dist = stats.ttest_ind(exec_times[in_id].numpy(), pg_times,
                                        alternative='greater')
        except RuntimeWarning:
            pg_dist = 1.0
        if math.isnan(pg_dist):
            pg_dist = 1.0
        try:
            _, conf_dist = stats.ttest_ind(pg_times, exec_times[out_id].numpy(),
                                        alternative='greater')
        except RuntimeWarning:
            conf_dist = 1.0
        if math.isnan(conf_dist):
            conf_dist = 1.0
        
        if out_dist < 0.025:
            all_query_enc.append(query_enc.view(1, -1))
            all_input_enc.append(plan_enc[in_id].view(1, -1))
            all_out_enc.append(plan_enc[out_id].view(1, -1))
            all_pg_dist.append(torch.tensor([pg_dist]).float().view(1, -1))
            all_out_dist.append(torch.tensor([out_dist]).float().view(1, -1))
            all_conf_dist.append(torch.tensor([conf_dist]).float().view(1, -1))

query_enc = torch.cat(all_query_enc)
input_enc = torch.cat(all_input_enc)
out_enc = torch.cat(all_out_enc)
pg_dist = torch.cat(all_pg_dist)
out_dist = torch.cat(all_out_dist)
conf_dist = torch.cat(all_conf_dist)

out_dist = (torch.exp(out_dist) - 1) / (np.exp(0.025) - 1)
pg_dist = (torch.exp(pg_dist) - 1) / (np.exp(1) - 1)
conf_dist = (torch.exp(conf_dist) - 1) / (np.exp(1) - 1)

train_data = torch.utils.data.TensorDataset(input_enc, query_enc, out_enc, pg_dist, out_dist, conf_dist)

In [ ]:
for test_query in tqdm(test_queries, desc="Processing test queries ...", total=len(test_queries)):
    # counter = 0
    query_enc = torch.load(os.path.join(DEMO_BASE_PATH, SCHEMA_FOLDER, DEMO_TYPE, test_query, 'query_enc.pt'))
    plan_enc = torch.load(os.path.join(DEMO_BASE_PATH, SCHEMA_FOLDER, DEMO_TYPE, test_query, 'plan_enc.pt'))
    exec_times = torch.load(os.path.join(DEMO_BASE_PATH, SCHEMA_FOLDER, DEMO_TYPE, test_query, 'exec_times.pt'))
    exec_time = torch.mean(exec_times, axis=1)
    exec_time, indices = torch.sort(exec_time, descending=True)
    plan_enc = plan_enc[indices]
    
    pg_times = pg_df[pg_df.query_ident == f'{test_query}'].execution_time.values
    
    for in_id, out_id in itertools.combinations(np.arange(0, len(exec_time) - 1), 2):
        # Problem with calculating p-values when data is close to identical
        try:
            _, out_dist = stats.ttest_ind(exec_times[in_id].numpy(), exec_times[out_id].numpy(),
                                        alternative='greater')
        except RuntimeWarning:
            out_dist = 1.0
        if math.isnan(out_dist):
            out_dist = 1.0
        try:
            _, pg_dist = stats.ttest_ind(exec_times[in_id].numpy(), pg_times,
                                        alternative='greater')
        except RuntimeWarning:
            pg_dist = 1.0
        if math.isnan(pg_dist):
            pg_dist = 1.0
        try:
            _, conf_dist = stats.ttest_ind(pg_times, exec_times[out_id].numpy(),
                                        alternative='greater')
        except RuntimeWarning:
            conf_dist = 1.0
        if math.isnan(conf_dist):
            conf_dist = 1.0
        
        if out_dist < 0.025:
            all_query_enc.append(query_enc.view(1, -1))
            all_input_enc.append(plan_enc[in_id].view(1, -1))
            all_out_enc.append(plan_enc[out_id].view(1, -1))
            all_pg_dist.append(torch.tensor([pg_dist]).float().view(1, -1))
            all_out_dist.append(torch.tensor([out_dist]).float().view(1, -1))
            all_conf_dist.append(torch.tensor([conf_dist]).float().view(1, -1))

query_enc = torch.cat(all_query_enc)
input_enc = torch.cat(all_input_enc)
out_enc = torch.cat(all_out_enc)
pg_dist = torch.cat(all_pg_dist)
out_dist = torch.cat(all_out_dist)
conf_dist = torch.cat(all_conf_dist)

out_dist = (torch.exp(out_dist) - 1) / (np.exp(0.025) - 1)
pg_dist = (torch.exp(pg_dist) - 1) / (np.exp(1) - 1)
conf_dist = (torch.exp(conf_dist) - 1) / (np.exp(1) - 1)

test_data = torch.utils.data.TensorDataset(input_enc, query_enc, out_enc, pg_dist, out_dist, conf_dist)

# Optimization with Optuna

Inspired with: https://github.com/elena-ecn/optuna-optimization-for-PyTorch-CNN/blob/main/optuna_optimization.py

In [ ]:
def train(epoch, train_loader, model, optimizerG, loss_G):
    model.train()
    G_loss = 0
    for batch_idx, (i_enc, q_enc, o_enc, p_dist, o_dist, c_dist) in enumerate(train_loader):
        i_enc, q_enc, o_enc, p_dist, o_dist, c_dist = i_enc.to(device), q_enc.to(device), o_enc.to(device), p_dist.to(device), o_dist.to(device), c_dist.to(device)
        ## Updating generator ##
        optimizerG.zero_grad()
        o_pred, mu, logvar, z = model(i_enc, q_enc, c_dist, o_dist)
        loss, kl = loss_function(loss_G, o_pred, o_enc, mu, logvar)
        errG = loss + kl
        errG.backward()
        G_loss += errG.detach().cpu().numpy()
        optimizerG.step()
    if epoch % 5 == 0:
        print(f'Epoch:{epoch} G Loss:{(G_loss / len(train_loader.dataset)):.5f}}')

In [ ]:
def loss_function(loss_fn, pred, out, mu, logvar):
    """
    Reconstruction + KL divergence losses summed over all elements and batch
    
    See Appendix B from VAE paper:
    Kingma and Welling. Auto-Encoding Variational Bayes. ICLR, 2014
    https://arxiv.org/abs/1312.6114
    0.5 * sum(1 + log(sigma^2) - mu^2 - sigma^2)
    """
    mask = (out > 0).detach()
    out[~mask] = float('-inf')
    ce = loss_fn(pred, softmax(out))
    std = torch.exp(0.5*logvar)
    KLD_latent = kld_gaussians(mu, std, 0, 1)
    return ce, KLD_latent

In [ ]:
def test(test_loader, model, loss_G):
    """
    Tests the model.
    """
    model.eval()  # Set the module in evaluation mode (only affects certain modules)
    model.to('cpu')
    avg_loss = 0
    with torch.no_grad():  # Disable gradient calculation
        n_iter = 0
        for batch_idx, (i_enc, q_enc, o_enc, p_dist, o_dist, c_dist) in enumerate(test_loader):
            i_enc, q_enc, o_enc, p_dist, o_dist, c_dist = i_enc.to(device), q_enc.to(device), o_enc.to(device), p_dist.to(device), o_dist.to(device), c_dist.to(device)
            ## Updating generator ##
            o_pred, mu, logvar, z = model(i_enc, q_enc, c_dist, o_dist)
            loss, kl = loss_function(loss_G, o_pred, o_enc, mu, logvar)
            errG = loss + kl
            avg_loss += errG..data.item()
            n_iter += 1
    avg_loss /= n_iter

    return avg_loss

In [ ]:
 config = {'join_order_benchmark': {
                    'NUM_POSSIBLE_JOINS': 53,
                    },
              'stack': {
                    'NUM_POSSIBLE_JOINS': 32,
              }
    }

In [ ]:
def objective(trial):
    """Objective function to be optimized by Optuna.
    Hyperparameters chosen to be optimized: optimizer, learning rate,
    dropout values, number of convolutional layers, number of filters of
    convolutional layers, number of neurons of fully connected layers.
    Inputs:
        - trial (optuna.trial._trial.Trial): Optuna trial
    Returns:
        - accuracy(torch.Tensor): The test accuracy. Parameter to be minimized.
    """

    # Define range of values to be tested for the hyperparameters
    latent_size = trial.suggest_int("latent_size", 40, 100)  # Number of neurons in bottleneck layer
    layers_size = [int(trial.suggest_int("layer_size_" + str(i), 40, 100))
                   for i in range(2)]  # Number of neurons in hidden layers
    step_size = trial.suggest_int("step_size", 10, 100)  # Step size of learning rate scheduler
    train_ratio = trial.suggest_float("train_ratio", 0.01, 0.99)  # the percentage of data to be feed
    batch_size = trial.suggest_int("batch_size", 25, 200)  # batch_size
    epochs = trial.suggest_int("epochs", 25, 200)  # number of epochs
    
    # Generate the model
    model = CVAE_G(config[WORKLOAD_NAME]['NUM_POSSIBLE_JOINS'] * NUM_JOIN_TYPES, 
                 latent_size,
                 layers_size,
                 config[WORKLOAD_NAME]['NUM_POSSIBLE_JOINS'] * QUERY_ENCODING_SIZE).to(device)

    # Generate the optimizers
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "RMSprop"])  # Optimizers
    lr = trial.suggest_float("lr", 1e-6, 1e-3)  # Learning rates
    optimizerG = getattr(optim, optimizer_name)(model.parameters(), lr=lr)
    schedulerG = StepLR(optimizerG, step_size=step_size, gamma=0.75)
    ce_loss = nn.CrossEntropyLoss()

    train_len = int(len(train_data)*train_ratio)
    train_data, _ = random_split(train_data, [train_len, len(train_data) - train_len])
    train_dataloader = torch.utils.data.DataLoader(train_data, 
                                                   batch_size=batch_size,
                                                   shuffle=True, drop_last=True)
    test_dataloader = torch.utils.data.DataLoader(test_data, 
                                                   batch_size=1,
                                                   shuffle=True, drop_last=True)

    # Training of the model
    for epoch in range(epochs):
        train(epoch, train_dataloader, model, optimizerG, ce_loss)
        schedulerG.step()
        loss = test(test_dataloader, model, ce_loss)  # Evaluate the model

        # For pruning (stops trial early if not promising)
        trial.report(loss, epoch)
        # Handle pruning based on the intermediate value.
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return loss

In [ ]:
number_of_trials = 100  # Number of Optuna trials
limit_obs = False  # Limit number of observations for faster computation

# Make runs repeatable
random_seed = 518
torch.backends.cudnn.enabled = False  # Disable cuDNN use of nondeterministic algorithms
torch.manual_seed(random_seed)
np.random.seed(random_seed)

In [ ]:
# Create an Optuna study to maximize test accuracy
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=number_of_trials, n_jobs=1)

# -------------------------------------------------------------------------
# Results
# -------------------------------------------------------------------------

# Find number of pruned and completed trials
pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

# Display the study statistics
print("\nStudy statistics: ")
print("  Number of finished trials: ", len(study.trials))
print("  Number of pruned trials: ", len(pruned_trials))
print("  Number of complete trials: ", len(complete_trials))

trial = study.best_trial
print("Best trial:")
print("  Value: ", trial.value)
print("  Params: ")
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))

# Save results to csv file
df = study.trials_dataframe().drop(['datetime_start', 'datetime_complete', 'duration'], axis=1)
df = df.loc[df['state'] == 'COMPLETE']        # Keep only results that did not prune
df = df.drop('state', axis=1)                 # Exclude state column
df = df.sort_values('value')                  # Sort based on accuracy
df.to_csv('optuna_results.csv', index=False)  # Save to csv file

# Display results in a dataframe
print("\nOverall Results (ordered by accuracy):\n {}".format(df))

# Find the most important hyperparameters
most_important_parameters = optuna.importance.get_param_importances(study, target=None)

# Display the most important hyperparameters
print('\nMost important hyperparameters:')
for key, value in most_important_parameters.items():
    print('  {}:{}{:.2f}%'.format(key, (15-len(key))*' ', value*100))

In [ ]:
trial.params